CELDA 1 — Librerías

In [ ]:
# =====================================================# 07_model_training# Modelo Predictivo Churn B2B# =====================================================from pyspark.sql import functions as Ffrom pyspark.ml import Pipelinefrom pyspark.ml.feature import (    StringIndexer,    VectorAssembler)from pyspark.ml.classification import GBTClassifierfrom pyspark.ml.evaluation import (    BinaryClassificationEvaluator,    MulticlassClassificationEvaluator)print("Librerías cargadas correctamente")

CELDA 2 — Leer Gold Dataset

In [0]:

gold_dataset = spark.read \
    .format("delta") \
    .load("/Volumes/workspace/default/churn_b2b/Gold/gold_dataset")

print("Registros:", gold_dataset.count())

display(gold_dataset.limit(5))

Registros: 5000


company_id,ruc,razon_social,sector,region,empleados,segmento,fecha_alta,ejecutivo_comercial,nps,csat,facturacion_total,facturacion_promedio,facturacion_maxima,dias_mora_promedio,deuda_total,trafico_total_gb,trafico_promedio_gb,ancho_banda_promedio,pico_maximo_mbps,cantidad_servicios,monto_mensual_promedio,sla_promedio,cantidad_tickets,tiempo_resolucion_promedio,tickets_escalados,cantidad_eventos,duracion_promedio_eventos,churn,risk_score
13,20303911718,Obras Capital EIRL,Construcci�n,Piura,7971,SMB,2018-01-06,Ejecutivo_14,85,4.7,741725.8500000001,30905.243750000005,58696.81,4.0,440142.5999999999,241287.69,10053.65375,402.1466666666667,1386.32,2,10260.905,99.9,17,26.529411764705884,1,49,27.3265306122449,0,17.92
30,20158692322,Software Andes SAC,Tecnolog�a,Ica,3189,Enterprise,2024-04-21,Ejecutivo_31,70,4.05,822160.07,34256.66958333333,59850.01,3.8333333333333335,555988.37,258854.73999999996,10785.614166666664,431.4241666666666,1487.25,1,11991.41,99.5,7,21.0,0,56,32.0,0,43.32
37,20777387214,Farmac�utica Capital EIRL,Salud,Cajamarca,316,Mid-Market,2022-01-04,Ejecutivo_38,65,4.25,595419.05,24809.127083333336,58741.98,4.25,730104.2499999999,76114.36,3171.431666666667,126.85708333333334,437.32,1,20400.13,99.5,11,27.90909090909091,1,57,28.75438596491228,0,16.45
41,20277434873,Academia Capital EIRL,Educaci�n,Tacna,29,Enterprise,2018-12-30,Ejecutivo_42,81,4.18,691905.09,28829.37875,58479.42,4.958333333333333,613774.9999999999,197289.32,8220.388333333334,328.8154166666667,1133.53,3,9986.08,99.76666666666667,16,29.0,0,48,32.916666666666664,0,19.54
58,20134352408,Manufactura Sur EIRL,Manufactura,Trujillo,1876,SMB,2025-05-03,Ejecutivo_9,52,4.54,773847.31,32243.63791666667,56065.09,4.291666666666667,535965.54,256234.27000000002,10676.427916666667,427.05500000000006,1472.2,2,9280.725,99.5,9,23.11111111111111,0,22,33.72727272727273,0,31.16


CELDA 3 — Validar columnas

In [0]:

print("Columnas del Gold Dataset")

for c in gold_dataset.columns:
    print(c)

Columnas del Gold Dataset
company_id
ruc
razon_social
sector
region
empleados
segmento
fecha_alta
ejecutivo_comercial
nps
csat
facturacion_total
facturacion_promedio
facturacion_maxima
dias_mora_promedio
deuda_total
trafico_total_gb
trafico_promedio_gb
ancho_banda_promedio
pico_maximo_mbps
cantidad_servicios
monto_mensual_promedio
sla_promedio
cantidad_tickets
tiempo_resolucion_promedio
tickets_escalados
cantidad_eventos
duracion_promedio_eventos
churn
risk_score


### CELDA 4 — Preparación de datos SIN fuga (split primero, indexado después)> **Corrección aplicada:** el indexado de variables categóricas y el ensamblado de> features ahora se realizan dentro de un `Pipeline` que se ajusta **solo con los datos> de entrenamiento**, después de dividir train/test. Así el conjunto de prueba permanece> completamente aislado y se evita la fuga de datos.

In [ ]:
# =====================================================# CELDA 4 - Base sin indexar (se indexará dentro del Pipeline)# =====================================================base = gold_dataset.select(    "company_id",    "empleados", "nps", "csat",    "facturacion_total", "facturacion_promedio", "facturacion_maxima",    "dias_mora_promedio", "deuda_total",    "trafico_total_gb", "trafico_promedio_gb", "ancho_banda_promedio",    "pico_maximo_mbps", "cantidad_servicios", "monto_mensual_promedio", "sla_promedio",    "cantidad_tickets", "tiempo_resolucion_promedio", "tickets_escalados",    "cantidad_eventos", "duracion_promedio_eventos",    "sector", "region", "segmento",    F.col("churn").cast("double").alias("label"))print("Base preparada:", base.count(), "registros")display(base.limit(5))

### CELDA 5 — Variables predictivas del modeloSe definen las 23 variables de entrada. `risk_score` **no** se incluye: es una salida delmodelo (probabilidad), no una entrada, por lo que su exclusión evita fuga de datos.

In [ ]:
# =====================================================# CELDA 5 - Variables del Modelo (23 features)# =====================================================feature_columns = [    "empleados", "nps", "csat",    "facturacion_total", "facturacion_promedio", "facturacion_maxima",    "dias_mora_promedio", "deuda_total",    "trafico_total_gb", "trafico_promedio_gb", "ancho_banda_promedio",    "pico_maximo_mbps", "cantidad_servicios", "monto_mensual_promedio", "sla_promedio",    "cantidad_tickets", "tiempo_resolucion_promedio", "tickets_escalados",    "cantidad_eventos", "duracion_promedio_eventos",    "sector_index", "region_index", "segmento_index"]print("Cantidad de variables:", len(feature_columns))print(feature_columns)

In [ ]:
# =====================================================# Verificación: risk_score NO entra como feature (anti fuga de datos)# =====================================================prohibidas = ["risk_score", "churn", "label"]fugas = [c for c in feature_columns if c in prohibidas]print("Variables prohibidas encontradas entre las features:", fugas)assert len(fugas) == 0, "FUGA DE DATOS: variable derivada de la etiqueta entre las features"print("Verificado: el modelo NO usa risk_score ni variables derivadas de la etiqueta.")

In [ ]:
# El schema final se verá tras aplicar el Pipeline (siguientes celdas)

### CELDA 6 — Split + Pipeline (indexado y ensamblado ajustados solo con train)

In [ ]:
# =====================================================# CELDA 6 - Split PRIMERO, luego Pipeline ajustado SOLO con train# =====================================================# 1) Dividir ANTES de cualquier ajuste (el test queda intacto)train_data, test_data = base.randomSplit([0.8, 0.2], seed=42)print("Train:", train_data.count(), "| Test:", test_data.count())# 2) Etapas del Pipelinesector_idx   = StringIndexer(inputCol="sector",   outputCol="sector_index",   handleInvalid="keep")region_idx   = StringIndexer(inputCol="region",   outputCol="region_index",   handleInvalid="keep")segmento_idx = StringIndexer(inputCol="segmento", outputCol="segmento_index", handleInvalid="keep")assembler = VectorAssembler(inputCols=feature_columns, outputCol="features", handleInvalid="keep")# 3) Ajustar el Pipeline SOLO con train, aplicarlo a ambosprep_pipeline = Pipeline(stages=[sector_idx, region_idx, segmento_idx, assembler])prep_model = prep_pipeline.fit(train_data)train_prepared = prep_model.transform(train_data).select("company_id", "features", "label")test_prepared  = prep_model.transform(test_data).select("company_id", "features", "label")print("Pipeline ajustado SOLO con datos de entrenamiento (sin fuga de datos)")display(train_prepared.limit(5))

> Las etapas de VectorAssembler y split ya se realizaron dentro del Pipeline anterior.

In [ ]:
# (Integrado en la CELDA 6 - Pipeline). Nada que ejecutar aquí.

### CELDA 7 — Entrenamiento del modelo

In [ ]:
# =====================================================# CELDA 7 - Entrenamiento (con el train preparado por el Pipeline)# =====================================================model = GBTClassifier(    labelCol="label",    featuresCol="features",    maxIter=100,    maxDepth=5,    seed=42)model = model.fit(train_prepared)print("Modelo entrenado correctamente")

### CELDA 8 — Predicciones sobre el test aislado

In [ ]:
# =====================================================# CELDA 8 - Predicciones# =====================================================predictions = model.transform(test_prepared)display(    predictions.select("company_id", "label", "prediction", "probability").limit(10))

> El modelo ya fue entrenado en la CELDA 7.

In [ ]:
# (Entrenamiento ya realizado en la CELDA 7). Nada que ejecutar aquí.

> Las predicciones ya se generaron en la CELDA 8.

In [ ]:
# (Predicciones ya generadas en la CELDA 8). Nada que ejecutar aquí.

In [0]:
from pyspark.sql.functions import col

# =====================================================
# INFORMACIÓN DE LA EMPRESA
# =====================================================

company_info = gold_dataset.select(

    "company_id",

    "ruc",

    "razon_social",

    "sector",

    "region",

    "segmento",

    "ejecutivo_comercial",

    "risk_score"

)

# =====================================================
# UNIR PREDICCIONES CON DATOS DE EMPRESA
# =====================================================

all_predictions = predictions.join(

    company_info,

    on="company_id",

    how="left"

).select(

    "company_id",

    "ruc",

    "razon_social",

    "sector",

    "region",

    "segmento",

    "ejecutivo_comercial",

    "label",

    "prediction",

    "probability",

    "risk_score"

)

display(all_predictions)

company_id,ruc,razon_social,sector,region,segmento,ejecutivo_comercial,label,prediction,probability,risk_score
3,20235116155,Express Pac�fico EIRL,Transporte,Ica,Enterprise,Ejecutivo_4,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9883839088126328"",""0.011616091187367172""]}",56.97
7,20030564139,Educativa Sur EIRL,Educaci�n,Trujillo,SMB,Ejecutivo_8,1.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.011616091187367266"",""0.9883839088126327""]}",77.03
9,20532871012,Agroindustrial Norte EIRL,Agroindustria,Ica,SMB,Ejecutivo_10,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9883839088126328"",""0.011616091187367172""]}",15.76
14,20824896383,Log�stica Sur EIRL,Transporte,Piura,Mid-Market,Ejecutivo_15,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9883839088126328"",""0.011616091187367172""]}",45.51
20,20267736026,Educativa Per� SAC,Educaci�n,Tacna,SMB,Ejecutivo_21,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9883839088126328"",""0.011616091187367172""]}",3.69
24,20435346247,Networks Norte EIRL,Telecomunicaciones,Trujillo,Enterprise,Ejecutivo_25,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9883839088126328"",""0.011616091187367172""]}",50.73
30,20158692322,Software Andes SAC,Tecnolog�a,Ica,Enterprise,Ejecutivo_31,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9883839088126328"",""0.011616091187367172""]}",43.32
36,20629946804,Retail Andes EIRL,Retail,Cajamarca,Mid-Market,Ejecutivo_37,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9883839088126328"",""0.011616091187367172""]}",47.09
46,20162720465,Educativa Inca SAC,Educaci�n,Chiclayo,Enterprise,Ejecutivo_47,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9883839088126328"",""0.011616091187367172""]}",46.3
47,20464170805,Manufactura Pac�fico EIRL,Manufactura,Lima,Enterprise,Ejecutivo_48,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9883839088126328"",""0.011616091187367172""]}",21.14


CELDA 12 — Accuracy

In [0]:

# =====================================================
# CELDA 12 - Accuracy
# =====================================================

accuracy = MulticlassClassificationEvaluator(

    labelCol="label",

    predictionCol="prediction",

    metricName="accuracy"

).evaluate(predictions)

print("Accuracy:", round(accuracy,4))

Accuracy: 1.0


Celda 13 Presición

In [0]:
precision = MulticlassClassificationEvaluator(

    labelCol="label",

    predictionCol="prediction",

    metricName="weightedPrecision"

).evaluate(predictions)

print("Precision:", round(precision,4))

Precision: 1.0


CELDA 14 — Recall

In [0]:

recall = MulticlassClassificationEvaluator(

    labelCol="label",

    predictionCol="prediction",

    metricName="weightedRecall"

).evaluate(predictions)

print("Recall:", round(recall,4))


Recall: 1.0


CELDA 15 — F1 Score

In [0]:

f1 = MulticlassClassificationEvaluator(

    labelCol="label",

    predictionCol="prediction",

    metricName="f1"

).evaluate(predictions)

print("F1 Score:", round(f1,4))

F1 Score: 1.0


### CELDA 15b — ROC-AUC y PR-AUC (métricas para clases desbalanceadas)> **Corrección aplicada:** se calcula el ROC-AUC comprometido como indicador de éxito> en la Semana 1, además del PR-AUC (más informativo con clases desbalanceadas).

In [ ]:
# =====================================================# CELDA 15b - ROC-AUC y PR-AUC# =====================================================roc_auc = BinaryClassificationEvaluator(    labelCol="label",    rawPredictionCol="rawPrediction",    metricName="areaUnderROC").evaluate(predictions)pr_auc = BinaryClassificationEvaluator(    labelCol="label",    rawPredictionCol="rawPrediction",    metricName="areaUnderPR").evaluate(predictions)print("ROC-AUC:", round(roc_auc, 4))print("PR-AUC :", round(pr_auc, 4))meta_roc = 0.80print("\nIndicador de éxito (ROC-AUC >= 0.80):",      "CUMPLIDO" if roc_auc >= meta_roc else "NO CUMPLIDO")

CELDA 16 — Matriz de Confusión

In [0]:

# =====================================================
# CELDA 16 - Matriz de Confusión
# =====================================================

from pyspark.sql import functions as F

tp = predictions.filter(
    (F.col("label") == 1) &
    (F.col("prediction") == 1)
).count()

tn = predictions.filter(
    (F.col("label") == 0) &
    (F.col("prediction") == 0)
).count()

fp = predictions.filter(
    (F.col("label") == 0) &
    (F.col("prediction") == 1)
).count()

fn = predictions.filter(
    (F.col("label") == 1) &
    (F.col("prediction") == 0)
).count()

print("========== MATRIZ DE CONFUSIÓN ==========")
print(f"TP (True Positive) : {tp}")
print(f"TN (True Negative) : {tn}")
print(f"FP (False Positive): {fp}")
print(f"FN (False Negative): {fn}")

========== MATRIZ DE CONFUSIÓN ==========
TP (True Positive) : 150
TN (True Negative) : 808
FP (False Positive): 0
FN (False Negative): 0


CELDA 17 — Guardar las métricas

Esta información será la que consumirá tu backend.

In [ ]:
# =====================================================# CELDA 17 - Crear DataFrame de Métricas (incluye ROC-AUC y PR-AUC)# =====================================================metrics = [(    float(accuracy), float(precision), float(recall), float(f1),    float(roc_auc), float(pr_auc),    int(tp), int(tn), int(fp), int(fn))]metrics_df = spark.createDataFrame(    metrics,    ["accuracy", "precision", "recall", "f1_score",     "roc_auc", "pr_auc",     "true_positive", "true_negative", "false_positive", "false_negative"])display(metrics_df)

In [0]:
gold_dataset.groupBy("churn").agg(

    F.avg("nps").alias("nps"),
    F.avg("csat").alias("csat"),
    F.avg("dias_mora_promedio").alias("mora"),
    F.avg("deuda_total").alias("deuda"),
    F.avg("cantidad_tickets").alias("tickets")

).show()

+-----+------------------+-----------------+------------------+-----------------+------------------+
|churn|               nps|             csat|              mora|            deuda|           tickets|
+-----+------------------+-----------------+------------------+-----------------+------------------+
|    1|20.047619047619047|2.001559523809525|32.658184523809524|602186.2978333329|51.430952380952384|
|    0| 74.76995192307692|4.255033653846156| 4.504667467948714| 602433.274932692|14.369711538461539|
+-----+------------------+-----------------+------------------+-----------------+------------------+



In [ ]:
all_predictions = model.transform(prep_model.transform(base).select("company_id","features","label"))

CELDA 18 — Predicción de las 5000 empresas

In [ ]:
# =====================================================# CELDA 18 - Predicciones para TODO el Dataset# =====================================================all_predictions = model.transform(prep_model.transform(base).select("company_id","features","label"))print("Total de predicciones:")print(all_predictions.count())display(    all_predictions.select(        "company_id",        "prediction",        "probability"    ).limit(10))

CELDA 19 — Recuperar información de las empresas

Ahora recuperamos las columnas descriptivas del Gold.

In [0]:

# =====================================================
# CELDA 19 - Información de Empresas
# =====================================================

company_info = gold_dataset.select(

    "company_id",

    "ruc",

    "razon_social",

    "sector",

    "region",

    "segmento",

    "ejecutivo_comercial"

)

display(company_info.limit(5))

company_id,ruc,razon_social,sector,region,segmento,ejecutivo_comercial
13,20303911718,Obras Capital EIRL,Construcci�n,Piura,SMB,Ejecutivo_14
30,20158692322,Software Andes SAC,Tecnolog�a,Ica,Enterprise,Ejecutivo_31
37,20777387214,Farmac�utica Capital EIRL,Salud,Cajamarca,Mid-Market,Ejecutivo_38
41,20277434873,Academia Capital EIRL,Educaci�n,Tacna,Enterprise,Ejecutivo_42
58,20134352408,Manufactura Sur EIRL,Manufactura,Trujillo,SMB,Ejecutivo_9


CELDA 20 — Obtener la probabilidad de Churn

Nos interesa únicamente la probabilidad de la clase 1 (Churn).

In [0]:


# =====================================================
# CELDA 20 - Risk Score
# =====================================================

from pyspark.sql.functions import udf

from pyspark.sql.types import DoubleType

extract_probability = udf(

    lambda v: float(v[1]),

    DoubleType()

)

predictions_final = all_predictions.withColumn(

    "risk_score",

    extract_probability("probability")

)

CELDA 21 — Unir con la información de las empresas

In [0]:
# =====================================================
# CELDA 21 - Join Final
# =====================================================

all_predictions = predictions_final.join(

    company_info,

    on="company_id",

    how="left"

)

display(all_predictions.limit(10))

company_id,features,label,rawPrediction,probability,prediction,risk_score,ruc,razon_social,sector,region,segmento,ejecutivo_comercial
13,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""7971.0"",""85.0"",""4.7"",""741725.8500000001"",""30905.243750000005"",""58696.81"",""4.0"",""440142.5999999999"",""241287.69"",""10053.65375"",""402.1466666666667"",""1386.32"",""2.0"",""10260.905"",""99.9"",""17.0"",""26.529411764705884"",""1.0"",""49.0"",""27.3265306122449"",""4.0"",""4.0"",""2.0""]}",0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.221839942871194"",""-2.221839942871194""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9883839088126328"",""0.011616091187367172""]}",0.0,0.011616091187367172,20303911718,Obras Capital EIRL,Construcci�n,Piura,SMB,Ejecutivo_14
30,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""3189.0"",""70.0"",""4.05"",""822160.07"",""34256.66958333333"",""59850.01"",""3.8333333333333335"",""555988.37"",""258854.73999999996"",""10785.614166666664"",""431.4241666666666"",""1487.25"",""1.0"",""11991.41"",""99.5"",""7.0"",""21.0"",""0.0"",""56.0"",""32.0"",""7.0"",""1.0"",""0.0""]}",0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.2218399428711946"",""-2.2218399428711946""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9883839088126328"",""0.011616091187367172""]}",0.0,0.011616091187367172,20158692322,Software Andes SAC,Tecnolog�a,Ica,Enterprise,Ejecutivo_31
37,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""316.0"",""65.0"",""4.25"",""595419.05"",""24809.127083333336"",""58741.98"",""4.25"",""730104.2499999999"",""76114.36"",""3171.431666666667"",""126.85708333333334"",""437.32"",""1.0"",""20400.13"",""99.5"",""11.0"",""27.90909090909091"",""1.0"",""57.0"",""28.75438596491228"",""6.0"",""9.0"",""1.0""]}",0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.221839942871194"",""-2.221839942871194""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9883839088126328"",""0.011616091187367172""]}",0.0,0.011616091187367172,20777387214,Farmac�utica Capital EIRL,Salud,Cajamarca,Mid-Market,Ejecutivo_38
41,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""29.0"",""81.0"",""4.18"",""691905.09"",""28829.37875"",""58479.42"",""4.958333333333333"",""613774.9999999999"",""197289.32"",""8220.388333333334"",""328.8154166666667"",""1133.53"",""3.0"",""9986.08"",""99.76666666666667"",""16.0"",""29.0"",""0.0"",""48.0"",""32.916666666666664"",""2.0"",""6.0"",""0.0""]}",0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.2218399428711946"",""-2.2218399428711946""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9883839088126328"",""0.011616091187367172""]}",0.0,0.011616091187367172,20277434873,Academia Capital EIRL,Educaci�n,Tacna,Enterprise,Ejecutivo_42
58,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""1876.0"",""52.0"",""4.54"",""773847.31"",""32243.63791666667"",""56065.09"",""4.291666666666667"",""535965.54"",""256234.27000000002"",""10676.427916666667"",""427.05500000000006"",""1472.2"",""2.0"",""9280.725"",""99.5"",""9.0"",""23.11111111111111"",""0.0"",""22.0"",""33.72727272727273"",""1.0"",""2.0"",""2.0""]}",0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.2218399428711946"",""-2.2218399428711946""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9883839088126328"",""0.011616091187367172""]}",0.0,0.011616091187367172,20134352408,Manufactura Sur EIRL,Manufactura,Trujillo,SMB,Ejecutivo_9
59,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""2265.0"",""6.0"",""1.47"",""747662.2099999998"",""31152.592083333326"",""58849.99"",""34.958333333333336"",""495281.9799999999"",""157935.56"",""6580.6483333333335"",""263.22583333333336"",""1154.5"",""5.0"",""6953.392"",""99.75"",""69.0"",""126.08695652173913"",""47.0"",""96.0"",""251.25"",""8.0"",""6.0"",""0.0""]}",1.0,"{""type"":""1"",""

CELDA 22 — Seleccionar columnas finales

In [0]:

# =====================================================
# CELDA 22 - Dataset Final
# =====================================================

all_predictions = all_predictions.select(

    "company_id",

    "ruc",

    "razon_social",

    "sector",

    "region",

    "segmento",

    "ejecutivo_comercial",

    "prediction",

    "risk_score"

)

display(all_predictions.limit(10))

company_id,ruc,razon_social,sector,region,segmento,ejecutivo_comercial,prediction,risk_score
13,20303911718,Obras Capital EIRL,Construcci�n,Piura,SMB,Ejecutivo_14,0.0,0.011616091187367172
30,20158692322,Software Andes SAC,Tecnolog�a,Ica,Enterprise,Ejecutivo_31,0.0,0.011616091187367172
37,20777387214,Farmac�utica Capital EIRL,Salud,Cajamarca,Mid-Market,Ejecutivo_38,0.0,0.011616091187367172
41,20277434873,Academia Capital EIRL,Educaci�n,Tacna,Enterprise,Ejecutivo_42,0.0,0.011616091187367172
58,20134352408,Manufactura Sur EIRL,Manufactura,Trujillo,SMB,Ejecutivo_9,0.0,0.011616091187367172
59,20271094777,Comercial Per� EIRL,Retail,Tacna,Enterprise,Ejecutivo_10,1.0,0.9883839088126327
60,20116719022,Transportes Pac�fico EIRL,Transporte,Lima,Mid-Market,Ejecutivo_11,0.0,0.011616091187367172
120,20418880592,Medical Andes SAC,Salud,Lima,Mid-Market,Ejecutivo_21,1.0,0.9883839088126327
153,20480162456,Producci�n Capital EIRL,Manufactura,Tacna,Enterprise,Ejecutivo_4,0.0,0.011616091187367172
159,20972896230,Campos Andes SAC,Agroindustria,Arequipa,SMB,Ejecutivo_10,0.0,0.011616091187367172


In [0]:
all_predictions.printSchema()

root
 |-- company_id: integer (nullable = true)
 |-- ruc: long (nullable = true)
 |-- razon_social: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- region: string (nullable = true)
 |-- segmento: string (nullable = true)
 |-- ejecutivo_comercial: string (nullable = true)
 |-- prediction: double (nullable = false)
 |-- risk_score: double (nullable = true)



CELDA 23 — Guardar en Delta

In [0]:
dbutils.fs.rm(
    "/Volumes/workspace/default/churn_b2b/Predictions/all_predictions",
    True
)

print("Carpeta eliminada")

Carpeta eliminada


In [0]:
# =====================================================
# CELDA 23 - Guardar Predicciones
# =====================================================
(
    all_predictions.write
    .format("delta")
    .mode("overwrite")
    .save("/Volumes/workspace/default/churn_b2b/Predictions/all_predictions")
)

print("Predicciones guardadas correctamente.")

Predicciones guardadas correctamente.


CELDA 24 — Verificación

In [0]:

# =====================================================
# CELDA 24 - Verificación
# =====================================================
predictions_saved = (
    spark.read
    .format("delta")
    .load("/Volumes/workspace/default/churn_b2b/Predictions/all_predictions")
)

print("Total de registros:", predictions_saved.count())

display(predictions_saved.limit(10))

Total de registros: 5000


company_id,ruc,razon_social,sector,region,segmento,ejecutivo_comercial,prediction,risk_score
13,20303911718,Obras Capital EIRL,Construcci�n,Piura,SMB,Ejecutivo_14,0.0,0.011616091187367172
30,20158692322,Software Andes SAC,Tecnolog�a,Ica,Enterprise,Ejecutivo_31,0.0,0.011616091187367172
37,20777387214,Farmac�utica Capital EIRL,Salud,Cajamarca,Mid-Market,Ejecutivo_38,0.0,0.011616091187367172
41,20277434873,Academia Capital EIRL,Educaci�n,Tacna,Enterprise,Ejecutivo_42,0.0,0.011616091187367172
58,20134352408,Manufactura Sur EIRL,Manufactura,Trujillo,SMB,Ejecutivo_9,0.0,0.011616091187367172
59,20271094777,Comercial Per� EIRL,Retail,Tacna,Enterprise,Ejecutivo_10,1.0,0.9883839088126327
60,20116719022,Transportes Pac�fico EIRL,Transporte,Lima,Mid-Market,Ejecutivo_11,0.0,0.011616091187367172
120,20418880592,Medical Andes SAC,Salud,Lima,Mid-Market,Ejecutivo_21,1.0,0.9883839088126327
153,20480162456,Producci�n Capital EIRL,Manufactura,Tacna,Enterprise,Ejecutivo_4,0.0,0.011616091187367172
159,20972896230,Campos Andes SAC,Agroindustria,Arequipa,SMB,Ejecutivo_10,0.0,0.011616091187367172


CELDA 25 — Convertir a Pandas

In [0]:

# =====================================================
# CELDA 25 - Convertir a Pandas
# =====================================================

pdf = predictions_saved.toPandas()

print(pdf.shape)

pdf.head()

(5000, 9)


,company_id,ruc,razon_social,sector,region,segmento,ejecutivo_comercial,prediction,risk_score
0,13,20303911718,Obras Capital EIRL,Construcci�n,Piura,SMB,Ejecutivo_14,0.0,0.011616
1,30,20158692322,Software Andes SAC,Tecnolog�a,Ica,Enterprise,Ejecutivo_31,0.0,0.011616
2,37,20777387214,Farmac�utica Capital EIRL,Salud,Cajamarca,Mid-Market,Ejecutivo_38,0.0,0.011616
3,41,20277434873,Academia Capital EIRL,Educaci�n,Tacna,Enterprise,Ejecutivo_42,0.0,0.011616
4,58,20134352408,Manufactura Sur EIRL,Manufactura,Trujillo,SMB,Ejecutivo_9,0.0,0.011616


CELDA 27 — Guardar Métricas

In [0]:

# =====================================================
# CELDA 27 - Guardar Métricas
# =====================================================
(
    metrics_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save("/Volumes/workspace/default/churn_b2b/Metrics/model_metrics")
)

print("Métricas guardadas correctamente.")


Métricas guardadas correctamente.


In [0]:
print("========== VARIABLES DISPONIBLES ==========")

variables = [
    "all_predictions",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "f1_score",
    "tp",
    "tn",
    "fp",
    "fn",
    "model",
    "feature_columns",
    "assembler",
    "gold_dataset"
]

for var in variables:
    print(f"{var}: {'✅' if var in globals() else '❌'}")

========== VARIABLES DISPONIBLES ==========
all_predictions: ✅
accuracy: ✅
precision: ✅
recall: ✅
f1: ✅
f1_score: ❌
tp: ✅
tn: ✅
fp: ✅
fn: ✅
model: ✅
feature_columns: ✅
assembler: ✅
gold_dataset: ✅


In [0]:
all_predictions.printSchema()

root
 |-- company_id: integer (nullable = true)
 |-- ruc: long (nullable = true)
 |-- razon_social: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- region: string (nullable = true)
 |-- segmento: string (nullable = true)
 |-- ejecutivo_comercial: string (nullable = true)
 |-- prediction: double (nullable = false)
 |-- risk_score: double (nullable = true)



In [0]:
display(predictions.limit(5))

company_id,features,label,rawPrediction,probability,prediction
3,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""4178.0"",""62.0"",""4.37"",""778626.56"",""32442.773333333334"",""58383.35"",""4.25"",""740424.2599999999"",""155663.63999999998"",""6485.985"",""259.43958333333336"",""894.37"",""4.0"",""11166.315"",""99.825"",""21.0"",""23.523809523809526"",""0.0"",""49.0"",""30.897959183673468"",""5.0"",""1.0"",""0.0""]}",0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.2218399428711946"",""-2.2218399428711946""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9883839088126328"",""0.011616091187367172""]}",0.0
7,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""6784.0"",""33.0"",""1.41"",""826078.1900000002"",""34419.92458333334"",""59511.15"",""40.958333333333336"",""746252.6399999999"",""73756.30999999998"",""3073.1795833333326"",""122.92666666666668"",""539.15"",""3.0"",""12257.313333333334"",""99.64999999999999"",""46.0"",""126.17391304347827"",""37.0"",""139.0"",""273.66906474820144"",""2.0"",""2.0"",""2.0""]}",1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""-2.2218399428711915"",""2.2218399428711915""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.011616091187367266"",""0.9883839088126327""]}",1.0
9,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""7973.0"",""75.0"",""4.94"",""766062.1699999999"",""31919.25708333333"",""59514.6"",""4.5"",""656301.31"",""151451.09"",""6310.462083333333"",""252.4183333333333"",""870.16"",""4.0"",""15258.9525"",""99.725"",""9.0"",""23.333333333333332"",""0.0"",""31.0"",""26.967741935483872"",""0.0"",""1.0"",""2.0""]}",0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.2218399428711946"",""-2.2218399428711946""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9883839088126328"",""0.011616091187367172""]}",0.0
14,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""6976.0"",""50.0"",""4.62"",""753898.2799999999"",""31412.42833333333"",""53446.07"",""4.708333333333333"",""635838.26"",""92449.47999999998"",""3852.061666666666"",""154.08208333333332"",""531.17"",""2.0"",""12507.565"",""99.95"",""10.0"",""25.3"",""1.0"",""45.0"",""28.4"",""5.0"",""4.0"",""1.0""]}",0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.221839942871194"",""-2.221839942871194""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9883839088126328"",""0.011616091187367172""]}",0.0
20,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""4880.0"",""67.0"",""4.0"",""644284.9599999998"",""26845.20666666666"",""48486.9"",""4.541666666666667"",""558807.9500000002"",""163004.08"",""6791.836666666666"",""271.6733333333333"",""936.54"",""5.0"",""14686.078000000003"",""99.85"",""14.0"",""22.571428571428573"",""0.0"",""59.0"",""26.694915254237287"",""2.0"",""6.0"",""2.0""]}",0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.2218399428711946"",""-2.2218399428711946""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9883839088126328"",""0.011616091187367172""]}",0.0


In [0]:
display(predictions.limit(5))

company_id,features,label,rawPrediction,probability,prediction
3,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""4178.0"",""62.0"",""4.37"",""778626.56"",""32442.773333333334"",""58383.35"",""4.25"",""740424.2599999999"",""155663.63999999998"",""6485.985"",""259.43958333333336"",""894.37"",""4.0"",""11166.315"",""99.825"",""21.0"",""23.523809523809526"",""0.0"",""49.0"",""30.897959183673468"",""5.0"",""1.0"",""0.0""]}",0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.2218399428711946"",""-2.2218399428711946""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9883839088126328"",""0.011616091187367172""]}",0.0
7,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""6784.0"",""33.0"",""1.41"",""826078.1900000002"",""34419.92458333334"",""59511.15"",""40.958333333333336"",""746252.6399999999"",""73756.30999999998"",""3073.1795833333326"",""122.92666666666668"",""539.15"",""3.0"",""12257.313333333334"",""99.64999999999999"",""46.0"",""126.17391304347827"",""37.0"",""139.0"",""273.66906474820144"",""2.0"",""2.0"",""2.0""]}",1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""-2.2218399428711915"",""2.2218399428711915""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.011616091187367266"",""0.9883839088126327""]}",1.0
9,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""7973.0"",""75.0"",""4.94"",""766062.1699999999"",""31919.25708333333"",""59514.6"",""4.5"",""656301.31"",""151451.09"",""6310.462083333333"",""252.4183333333333"",""870.16"",""4.0"",""15258.9525"",""99.725"",""9.0"",""23.333333333333332"",""0.0"",""31.0"",""26.967741935483872"",""0.0"",""1.0"",""2.0""]}",0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.2218399428711946"",""-2.2218399428711946""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9883839088126328"",""0.011616091187367172""]}",0.0
14,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""6976.0"",""50.0"",""4.62"",""753898.2799999999"",""31412.42833333333"",""53446.07"",""4.708333333333333"",""635838.26"",""92449.47999999998"",""3852.061666666666"",""154.08208333333332"",""531.17"",""2.0"",""12507.565"",""99.95"",""10.0"",""25.3"",""1.0"",""45.0"",""28.4"",""5.0"",""4.0"",""1.0""]}",0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.221839942871194"",""-2.221839942871194""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9883839088126328"",""0.011616091187367172""]}",0.0
20,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""4880.0"",""67.0"",""4.0"",""644284.9599999998"",""26845.20666666666"",""48486.9"",""4.541666666666667"",""558807.9500000002"",""163004.08"",""6791.836666666666"",""271.6733333333333"",""936.54"",""5.0"",""14686.078000000003"",""99.85"",""14.0"",""22.571428571428573"",""0.0"",""59.0"",""26.694915254237287"",""2.0"",""6.0"",""2.0""]}",0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.2218399428711946"",""-2.2218399428711946""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9883839088126328"",""0.011616091187367172""]}",0.0


In [0]:
predictions.printSchema()

root
 |-- company_id: integer (nullable = true)
 |-- features: vectorudt (nullable = true)
 |-- label: double (nullable = true)
 |-- rawPrediction: vectorudt (nullable = true)
 |-- probability: vectorudt (nullable = true)
 |-- prediction: double (nullable = false)

